# Desempenho Escolar no Brasil (2015–2024) — Notebook de Análise

**Disciplina:** Linguagem de Programação — Análise e Visualização de Dados com Python
**Avaliação:** G1 · Tema 24 — Desempenho Escolar no Brasil
**Tecnologias:** Python · Pandas · NumPy · Matplotlib · Seaborn · SQLAlchemy/SQLite · Streamlit (dashboard)

Este notebook documenta todo o processo analítico do projeto: da leitura da base ao cálculo dos KPIs, passando pela limpeza, engenharia de atributos, análise exploratória, visualizações e interpretação. O dashboard interativo (`app.py` + `pages/`) reutiliza as mesmas funções, que ficam no pacote `utils/`.

## 1. Introdução

O desempenho escolar é um indicador central da qualidade da educação. Ele influencia diretamente a formação profissional, a inclusão social, o desenvolvimento econômico e a redução das desigualdades. Medir e comparar o desempenho entre regiões, estados, redes de ensino e disciplinas permite identificar padrões, localizar regiões críticas e orientar políticas públicas.

**Objetivo:** investigar indicadores de desempenho escolar no Brasil entre 2015 e 2024 a partir de uma base simulada, respondendo às perguntas orientadoras do projeto:

1. Quais estados apresentam melhor desempenho escolar?
2. Existem diferenças entre escolas públicas e privadas?
3. Houve melhoria no desempenho ao longo do tempo?
4. Existe relação entre renda e desempenho?
5. Como o acesso à internet impacta o aprendizado?
6. Quais disciplinas apresentam menor rendimento?
7. Existem desigualdades regionais relevantes?

## 2. Contextualização educacional

No Brasil, avaliações como o SAEB e o IDEB mostram historicamente três padrões: (i) desigualdade regional, com Sul e Sudeste à frente de Norte e Nordeste; (ii) diferença de resultados entre redes pública e privada, fortemente associada ao nível socioeconômico das famílias; e (iii) rendimento mais baixo em Matemática do que em Língua Portuguesa, especialmente no Ensino Médio. Fatores como renda familiar, escolaridade dos pais, infraestrutura escolar e, mais recentemente, acesso à internet aparecem na literatura como variáveis associadas ao desempenho.

Esses padrões servem de **referência conceitual** para interpretar a base simulada: onde a simulação reproduz esses padrões, a leitura é direta; onde não reproduz, o resultado é discutido como limitação da base, não como fato educacional.

## 3. Explicação da base de dados

Arquivo: `dados/simulacao_desempenho_escolar_brasil.csv` (dataset simulado fornecido pela disciplina).

| Coluna | Descrição |
|---|---|
| `ano`, `semestre`, `data` | Ano letivo (2015–2024), semestre (1 ou 2) e data de referência |
| `regiao`, `uf`, `municipio` | Localização: região, estado e município |
| `rede_ensino` | Pública ou Privada |
| `disciplina` | Matemática, Português, Ciências, História, Geografia |
| `media_notas` | Média geral das notas (0–100) |
| `taxa_aprovacao`, `taxa_reprovacao` | Percentuais de aprovação e reprovação |
| `acesso_internet` | Percentual de acesso digital |
| `renda_media_familiar` | Renda média familiar (R$) |
| `indice_desempenho` | Índice geral de desempenho (0–100) |
| `nivel_desempenho` | Baixo, Médio, Alto, Excelente |

Cada linha é uma observação agregada: **município × rede × disciplina × semestre**.

## 4. Leitura dos dados

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep", font_scale=0.95)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PASTA_IMAGENS = RAIZ / "imagens"
PASTA_IMAGENS.mkdir(exist_ok=True)

CORES_REGIAO = {"Norte": "#2A9D8F", "Nordeste": "#E76F51", "Centro-Oeste": "#E9C46A", "Sudeste": "#264653", "Sul": "#8AB17D"}
CORES_REDE = {"Pública": "#264653", "Privada": "#E76F51"}
ORDEM_REGIAO = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]
ORDEM_NIVEL = ["Baixo", "Médio", "Alto", "Excelente"]

def salvar(fig, nome):
    caminho = PASTA_IMAGENS / nome
    fig.savefig(caminho, dpi=130, bbox_inches="tight", facecolor="white")
    print(f"figura salva em imagens/{nome}")

In [2]:
bruto = pd.read_csv(RAIZ / "dados" / "simulacao_desempenho_escolar_brasil.csv", encoding="utf-8-sig")
print(f"Dimensões: {bruto.shape[0]} linhas x {bruto.shape[1]} colunas")
bruto.head()

Dimensões: 740 linhas x 15 colunas


,ano,semestre,data,regiao,uf,municipio,rede_ensino,disciplina,media_notas,taxa_aprovacao,taxa_reprovacao,acesso_internet,renda_media_familiar,indice_desempenho,nivel_desempenho
0,2015,1,2015-06-01,Norte,AM,Manaus,Pública,Matemática,38.05,81.01,17.27,82.1,4624.09,66.56,Excelente
1,2015,1,2015-06-01,Norte,PA,Belém,Privada,Geografia,61.51,87.30,1.13,92.3,3178.52,51.25,Baixo
2,2015,1,2015-06-01,Norte,PA,Santarém,Privada,Português,62.11,89.05,12.48,74.9,4316.80,52.29,Baixo
3,2015,1,2015-06-01,Norte,RO,Porto Velho,Pública,História,65.67,98.89,15.36,62.2,1792.32,42.05,Alto
4,2015,1,2015-06-01,Norte,TO,Palmas,Pública,Português,54.86,95.94,23.97,60.0,1629.89,86.67,Baixo


In [3]:
bruto.info()

<class 'pandas.DataFrame'>
RangeIndex: 740 entries, 0 to 739
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ano                   740 non-null    int64  
 1   semestre              740 non-null    int64  
 2   data                  740 non-null    str    
 3   regiao                740 non-null    str    
 4   uf                    740 non-null    str    
 5   municipio             740 non-null    str    
 6   rede_ensino           740 non-null    str    
 7   disciplina            740 non-null    str    
 8   media_notas           740 non-null    float64
 9   taxa_aprovacao        740 non-null    float64
 10  taxa_reprovacao       740 non-null    float64
 11  acesso_internet       740 non-null    float64
 12  renda_media_familiar  740 non-null    float64
 13  indice_desempenho     740 non-null    float64
 14  nivel_desempenho      740 non-null    str    
dtypes: float64(6), int64(2), str(7)
me

In [4]:
bruto.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
ano,740.0,2019.50,2.87,2015.00,2017.00,2019.50,2022.00,2024.00
semestre,740.0,1.50,0.50,1.00,1.00,1.50,2.00,2.00
media_notas,740.0,64.97,17.53,35.08,49.68,65.16,80.62,94.93
taxa_aprovacao,740.0,77.42,12.80,55.03,66.48,77.21,88.88,98.93
taxa_reprovacao,740.0,18.36,9.62,1.11,10.36,18.56,26.86,35.00
acesso_internet,740.0,67.41,18.55,35.00,51.28,67.40,83.60,98.00
renda_media_familiar,740.0,3742.12,1288.07,1500.71,2626.78,3764.12,4811.78,5987.62
indice_desempenho,740.0,64.86,17.01,35.01,50.61,63.75,80.58,95.00


In [5]:
for col in ["ano", "semestre", "regiao", "uf", "rede_ensino", "disciplina", "nivel_desempenho"]:
    print(f"{col:18s} ({bruto[col].nunique():2d} valores): {sorted(bruto[col].unique().tolist())}")

ano                (10 valores): [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
semestre           ( 2 valores): [1, 2]
regiao             ( 5 valores): ['Centro-Oeste', 'Nordeste', 'Norte', 'Sudeste', 'Sul']
uf                 (20 valores): ['AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PR', 'RJ', 'RO', 'RS', 'SC', 'SP', 'TO']
rede_ensino        ( 2 valores): ['Privada', 'Pública']
disciplina         ( 5 valores): ['Ciências', 'Geografia', 'História', 'Matemática', 'Português']
nivel_desempenho   ( 4 valores): ['Alto', 'Baixo', 'Excelente', 'Médio']


## 5. Limpeza e preparação

Verificações realizadas:

1. valores ausentes e duplicatas;
2. tipos de dados (datas e numéricos);
3. padronização de textos (espaços, maiúsculas);
4. faixas válidas dos percentuais (0–100);
5. **consistência interna**: `taxa_aprovacao + taxa_reprovacao` não deveria ultrapassar 100 %, e `nivel_desempenho` deveria acompanhar `indice_desempenho`.

In [6]:
print("Valores ausentes por coluna:")
print(bruto.isna().sum())
print(f"\nLinhas duplicadas: {bruto.duplicated().sum()}")

Valores ausentes por coluna:
ano                     0
semestre                0
data                    0
regiao                  0
uf                      0
municipio               0
rede_ensino             0
disciplina              0
media_notas             0
taxa_aprovacao          0
taxa_reprovacao         0
acesso_internet         0
renda_media_familiar    0
indice_desempenho       0
nivel_desempenho        0
dtype: int64

Linhas duplicadas: 0


In [7]:
soma_taxas = bruto["taxa_aprovacao"] + bruto["taxa_reprovacao"]
inconsistentes = (soma_taxas > 100).sum()
print(f"Registros com aprovação + reprovação > 100%: {inconsistentes} ({inconsistentes / len(bruto):.1%})")
print(f"Soma mínima: {soma_taxas.min():.1f} | máxima: {soma_taxas.max():.1f}")

print("\nÍndice de desempenho por nível informado (a faixa deveria crescer com o nível):")
print(bruto.groupby("nivel_desempenho")["indice_desempenho"].agg(["min", "mean", "max", "count"]).reindex(ORDEM_NIVEL).round(1))

Registros com aprovação + reprovação > 100%: 291 (39.3%)
Soma mínima: 57.7 | máxima: 133.6

Índice de desempenho por nível informado (a faixa deveria crescer com o nível):
                   min  mean   max  count
nivel_desempenho                         
Baixo             35.0  64.5  94.7    193
Médio             35.4  65.5  94.8    171
Alto              35.2  65.2  95.0    181
Excelente         35.1  64.3  94.7    195


**Diagnóstico:** a base não possui ausentes nem duplicatas, mas apresenta duas inconsistências típicas de dados simulados: cerca de 40 % dos registros têm aprovação + reprovação acima de 100 %, e o `nivel_desempenho` informado **não guarda relação** com o `indice_desempenho` (todos os níveis cobrem a faixa 35–95). O tratamento abaixo mantém as colunas originais, cria uma **flag de inconsistência** e um **nível recalculado** a partir do índice, para que a análise possa usar um campo coerente.

O pipeline é implementado em `utils/dados.py` (função `tratar_dados`) e reproduzido aqui passo a passo.

In [8]:
df = bruto.copy()

# 1) nomes de colunas e textos
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
for col in ["regiao", "uf", "municipio", "rede_ensino", "disciplina", "nivel_desempenho"]:
    df[col] = df[col].astype("string").str.strip()
df["uf"] = df["uf"].str.upper()
df["rede_ensino"] = df["rede_ensino"].str.capitalize()

# 2) tipos
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
df["semestre"] = pd.to_numeric(df["semestre"], errors="coerce").astype("Int64")
numericas = ["media_notas", "taxa_aprovacao", "taxa_reprovacao", "acesso_internet", "renda_media_familiar", "indice_desempenho"]
for col in numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 3) chaves obrigatórias, duplicatas e datas reconstruídas
df = df.dropna(subset=["ano", "semestre", "uf", "disciplina"]).drop_duplicates().reset_index(drop=True)
df["ano"] = df["ano"].astype(int); df["semestre"] = df["semestre"].astype(int)
data_reconstruida = pd.to_datetime({"year": df["ano"], "month": np.where(df["semestre"] == 1, 6, 12), "day": 1})
df["data"] = df["data"].fillna(data_reconstruida)

# 4) imputação (mediana por UF x disciplina) e limites de percentuais
for col in numericas:
    if df[col].isna().any():
        df[col] = df[col].fillna(df.groupby(["uf", "disciplina"])[col].transform("median")).fillna(df[col].median())
for col in ["taxa_aprovacao", "taxa_reprovacao", "acesso_internet"]:
    df[col] = df[col].clip(0, 100)

# 5) flag de inconsistência e categorias ordenadas
df["taxas_inconsistentes"] = (df["taxa_aprovacao"] + df["taxa_reprovacao"]) > 100
df["nivel_desempenho"] = pd.Categorical(df["nivel_desempenho"].astype(str), categories=ORDEM_NIVEL, ordered=True)
df["regiao"] = pd.Categorical(df["regiao"].astype(str), categories=ORDEM_REGIAO, ordered=True)

print(f"Após tratamento: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.dtypes

Após tratamento: 740 linhas x 16 colunas


ano                              int64
semestre                         int64
data                    datetime64[us]
regiao                        category
uf                              string
municipio                       string
rede_ensino                     string
disciplina                      string
media_notas                    float64
taxa_aprovacao                 float64
taxa_reprovacao                float64
acesso_internet                float64
renda_media_familiar           float64
indice_desempenho              float64
nivel_desempenho              category
taxas_inconsistentes              bool
dtype: object

## 6. Engenharia de atributos

Novas variáveis criadas para a análise:

- `periodo` — rótulo `AAAA/S1` ou `AAAA/S2`, usado em séries temporais e heatmaps;
- `nome_uf` — nome completo do estado;
- `faixa_renda` — quartis da renda média familiar (Q1 = menor renda);
- `faixa_internet` — faixas de acesso digital (até 50 %, 50–65 %, 65–80 %, acima de 80 %);
- `nivel_calculado` — nível de desempenho derivado do índice (< 50 Baixo, 50–65 Médio, 65–80 Alto, ≥ 80 Excelente);
- `score_composto` — média entre nota e índice de desempenho;
- `periodo_pandemia` — marcador para 2020–2021.

In [9]:
from utils.dados import NOMES_UF

df["periodo"] = df["ano"].astype(str) + "/S" + df["semestre"].astype(str)
df["nome_uf"] = df["uf"].map(NOMES_UF).fillna(df["uf"])
df["faixa_renda"] = pd.qcut(df["renda_media_familiar"], q=4, labels=["Q1 (menor renda)", "Q2", "Q3", "Q4 (maior renda)"])
df["faixa_internet"] = pd.cut(df["acesso_internet"], bins=[0, 50, 65, 80, 100], labels=["Até 50%", "50–65%", "65–80%", "Acima de 80%"], include_lowest=True)
df["nivel_calculado"] = pd.cut(df["indice_desempenho"], bins=[0, 50, 65, 80, 100], labels=ORDEM_NIVEL, include_lowest=True)
df["score_composto"] = ((df["media_notas"] + df["indice_desempenho"]) / 2).round(2)
df["periodo_pandemia"] = df["ano"].between(2020, 2021)

concordancia = (df["nivel_desempenho"].astype(str) == df["nivel_calculado"].astype(str)).mean()
print(f"Concordância entre nível informado e nível recalculado: {concordancia:.1%}")
print("\nDistribuição do nível recalculado:")
print(df["nivel_calculado"].value_counts().reindex(ORDEM_NIVEL))

# Confere que o pipeline do pacote utils produz o mesmo resultado usado pelo dashboard
from utils.dados import preparar_base
assert preparar_base().shape == df.shape, "pipeline do dashboard diverge do notebook"
print("\nPipeline do dashboard (utils.dados.preparar_base) confere com o notebook.")
df[["periodo", "nome_uf", "faixa_renda", "faixa_internet", "nivel_calculado", "score_composto", "periodo_pandemia"]].head()

Concordância entre nível informado e nível recalculado: 24.2%

Distribuição do nível recalculado:
nivel_calculado
Baixo        175
Médio        210
Alto         164
Excelente    191
Name: count, dtype: int64

Pipeline do dashboard (utils.dados.preparar_base) confere com o notebook.


,periodo,nome_uf,faixa_renda,faixa_internet,nivel_calculado,score_composto,periodo_pandemia
0,2015/S1,Amazonas,Q3,Acima de 80%,Alto,52.30,False
1,2015/S1,Pará,Q2,Acima de 80%,Médio,56.38,False
2,2015/S1,Pará,Q3,65–80%,Médio,57.20,False
3,2015/S1,Rondônia,Q1 (menor renda),50–65%,Baixo,53.86,False
4,2015/S1,Tocantins,Q1 (menor renda),50–65%,Excelente,70.76,False


In [10]:
# Base tratada salva para reuso (dashboard e banco de dados)
df.to_csv(RAIZ / "dados" / "desempenho_escolar_tratado.csv", index=False, encoding="utf-8-sig")
print("dados/desempenho_escolar_tratado.csv salvo")

dados/desempenho_escolar_tratado.csv salvo


### 6.1 Persistência em banco relacional (SQLAlchemy + SQLite)

Os dados tratados são gravados em `database/desempenho_escolar.db` com um esquema em estrela (`dim_regiao`, `dim_estado`, `dim_municipio`, `dim_disciplina`, `fato_avaliacao`) e uma visão `vw_desempenho`. O dashboard consulta esse banco na página *Explorador de Dados*.

In [11]:
from database.criar_banco import criar_banco, consultar, resumo_banco

caminho_db = criar_banco(df)
print(f"Banco criado: {caminho_db.relative_to(RAIZ)}")
display(resumo_banco())
consultar("""
    SELECT regiao, rede_ensino, ROUND(AVG(media_notas), 2) AS media_notas, COUNT(*) AS registros
    FROM vw_desempenho GROUP BY regiao, rede_ensino ORDER BY media_notas DESC
""")

Banco criado: database\desempenho_escolar.db


,tabela,linhas
0,dim_regiao,5
1,dim_estado,20
2,dim_municipio,37
3,dim_disciplina,5
4,fato_avaliacao,740


,regiao,rede_ensino,media_notas,registros
0,Centro-Oeste,Pública,69.42,54
1,Centro-Oeste,Privada,68.18,46
2,Sul,Pública,66.61,64
3,Sul,Privada,65.34,56
4,Nordeste,Pública,65.24,83
5,Nordeste,Privada,64.70,77
6,Sudeste,Privada,64.34,139
7,Norte,Privada,63.17,49
8,Norte,Pública,63.16,51
9,Sudeste,Pública,62.94,121


## 7. Análise exploratória

Distribuições, balanceamento das categorias e estatísticas por grupo.

In [12]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), numericas):
    sns.histplot(df[col], kde=True, ax=ax, color="#264653", bins=25)
    ax.axvline(df[col].mean(), color="#E76F51", linestyle="--", label=f"média {df[col].mean():.1f}")
    ax.set_title(col.replace("_", " ").capitalize(), fontweight="bold")
    ax.set_xlabel(""); ax.legend(frameon=False)
fig.suptitle("Distribuição das variáveis numéricas", fontsize=15, fontweight="bold", y=1.01)
fig.tight_layout()
salvar(fig, "01_distribuicoes.png")
plt.show()

figura salva em imagens/01_distribuicoes.png


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
df["regiao"].value_counts().reindex(ORDEM_REGIAO).plot.bar(ax=axes[0], color=[CORES_REGIAO[r] for r in ORDEM_REGIAO])
axes[0].set_title("Registros por região", fontweight="bold")
df["disciplina"].value_counts().plot.bar(ax=axes[1], color="#2A9D8F"); axes[1].set_title("Registros por disciplina", fontweight="bold")
df["rede_ensino"].value_counts().plot.bar(ax=axes[2], color=[CORES_REDE[r] for r in df["rede_ensino"].value_counts().index]); axes[2].set_title("Registros por rede", fontweight="bold")
for ax in axes:
    ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
    for p in ax.patches:
        ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=9)
fig.tight_layout()
salvar(fig, "02_balanceamento.png")
plt.show()

figura salva em imagens/02_balanceamento.png


In [14]:
resumo_regiao = df.groupby(["regiao", "rede_ensino"], observed=True)[numericas].mean().round(2)
resumo_regiao

media_notas  taxa_aprovacao  taxa_reprovacao  acesso_internet  renda_media_familiar  indice_desempenho
regiao       rede_ensino                                                                                                        
Norte        Privada            63.17           77.37            14.78            66.48               3555.00              63.67
             Pública            63.16           78.82            19.38            68.19               3810.05              70.51
Nordeste     Privada            64.70           78.32            18.77            65.39               3930.42              63.40
             Pública            65.24           76.39            19.04            70.28               3973.17              66.22
Centro-Oeste Privada            68.18           76.89            19.31            70.26               3578.05              66.86
             Pública            69.42           79.47            19.69            67.20               3597.63              66.41
Sudeste      Privada            64.34           77.69            17.57            66.92               3636.13              66.18
             Pública            62.94           77.32            19.74            63.84               3790.43              64.43
Sul          Privada            65.34           77.91            17.72            69.45               3782.23              63.02
             Pública            66.61           74.46            16.77            70.35               3648.66              58.10

## 8. KPIs

Indicadores exigidos pelo tema, calculados sobre a base completa (no dashboard eles reagem aos filtros).

In [15]:
from utils.dados import calcular_kpis

kpis = calcular_kpis(df)
tabela_kpis = pd.DataFrame([
    ["Média geral das notas", f"{kpis['media_notas']:.2f}", "Indicador educacional"],
    ["Taxa média de aprovação", f"{kpis['taxa_aprovacao']:.2f}%", "Indicador escolar"],
    ["Índice médio de desempenho", f"{kpis['indice_desempenho']:.2f}", "Indicador nacional"],
    ["Estado com melhor desempenho", f"{kpis['melhor_uf']} ({kpis['melhor_uf_valor']:.2f})", "Ranking regional (índice médio)"],
    ["Rede com maior desempenho", f"{kpis['melhor_rede']} ({kpis['melhor_rede_valor']:.2f})", "Pública x privada (nota média)"],
    ["Disciplina mais crítica", f"{kpis['disciplina_critica']} ({kpis['disciplina_critica_valor']:.2f})", "Menor rendimento (nota média)"],
], columns=["KPI", "Valor", "Descrição"])
tabela_kpis

,KPI,Valor,Descrição
0,Média geral das notas,64.97,Indicador educacional
1,Taxa média de aprovação,77.42%,Indicador escolar
2,Índice médio de desempenho,64.86,Indicador nacional
3,Estado com melhor desempenho,RO (70.49),Ranking regional (índice médio)
4,Rede com maior desempenho,Pública (65.05),Pública x privada (nota média)
5,Disciplina mais crítica,Português (63.13),Menor rendimento (nota média)


In [16]:
import json
(RAIZ / "dados" / "resumo_kpis.json").write_text(json.dumps(kpis, ensure_ascii=False, indent=2), encoding="utf-8")
print("dados/resumo_kpis.json salvo (usado pela página index.html)")

dados/resumo_kpis.json salvo (usado pela página index.html)


## 9. Visualizações

As seis visualizações obrigatórias do tema, construídas com Matplotlib e Seaborn e salvas em `imagens/` (usadas também na página HTML do projeto).

### 9.1 Linha temporal — evolução do desempenho

In [17]:
serie = df.groupby(["ano", "semestre", "periodo"], observed=True)[["media_notas", "indice_desempenho", "taxa_aprovacao"]].mean().reset_index().sort_values(["ano", "semestre"])
x = np.arange(len(serie))
coef = np.polyfit(x, serie["media_notas"], 1)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(serie["periodo"], serie["media_notas"], marker="o", color="#264653", linewidth=2.5, label="Média das notas")
ax.plot(serie["periodo"], serie["indice_desempenho"], marker="s", color="#2A9D8F", linewidth=2, label="Índice de desempenho")
ax.plot(serie["periodo"], np.polyval(coef, x), linestyle="--", color="#E76F51", label=f"Tendência das notas ({coef[0]:+.2f}/semestre)")
ax.axvspan(serie.index[serie["ano"] == 2020].min() - 0.5 if (serie["ano"] == 2020).any() else 0, serie.index[serie["ano"] == 2021].max() + 0.5 if (serie["ano"] == 2021).any() else 0, color="#E9C46A", alpha=0.15, label="Pandemia (2020–2021)")
ax.set_title("Evolução semestral do desempenho escolar (2015–2024)", loc="left", fontsize=14, fontweight="bold")
ax.set_xlabel("Período"); ax.set_ylabel("Pontos (0–100)")
ax.tick_params(axis="x", rotation=45); ax.legend(frameon=False, ncol=4, loc="lower left")
fig.tight_layout()
salvar(fig, "03_linha_temporal.png")
plt.show()

anual = df.groupby("ano")["media_notas"].mean()
print(f"Variação {anual.index[0]} -> {anual.index[-1]}: {anual.iloc[-1] - anual.iloc[0]:+.2f} pontos | melhor ano: {anual.idxmax()} ({anual.max():.2f}) | pior ano: {anual.idxmin()} ({anual.min():.2f})")

figura salva em imagens/03_linha_temporal.png
Variação 2015 -> 2024: +0.54 pontos | melhor ano: 2023 (67.90) | pior ano: 2018 (62.40)


### 9.2 Barras por estado — comparação regional

In [18]:
rank = df.groupby(["uf", "regiao"], observed=True)["indice_desempenho"].mean().reset_index().sort_values("indice_desempenho", ascending=True)
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(rank["uf"], rank["indice_desempenho"], color=[CORES_REGIAO[r] for r in rank["regiao"].astype(str)])
ax.axvline(df["indice_desempenho"].mean(), color="#555", linestyle=":", label=f"média nacional {df['indice_desempenho'].mean():.1f}")
for i, v in enumerate(rank["indice_desempenho"]):
    ax.text(v + 0.2, i, f"{v:.1f}", va="center", fontsize=9)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=r) for r, c in CORES_REGIAO.items()] + [plt.Line2D([0], [0], color="#555", linestyle=":", label=f"média {df['indice_desempenho'].mean():.1f}")], frameon=False, loc="lower right")
ax.set_title("Índice médio de desempenho por estado (2015–2024)", loc="left", fontsize=14, fontweight="bold")
ax.set_xlabel("Índice de desempenho"); ax.set_xlim(rank["indice_desempenho"].min() - 3, rank["indice_desempenho"].max() + 3)
fig.tight_layout()
salvar(fig, "04_barras_estado.png")
plt.show()
print("Top 3:", rank.tail(3)[::-1][["uf", "indice_desempenho"]].round(2).to_dict("records"))
print("Bottom 3:", rank.head(3)[["uf", "indice_desempenho"]].round(2).to_dict("records"))

figura salva em imagens/04_barras_estado.png
Top 3: [{'uf': 'RO', 'indice_desempenho': 70.49}, {'uf': 'PA', 'indice_desempenho': 69.59}, {'uf': 'BA', 'indice_desempenho': 68.71}]
Bottom 3: [{'uf': 'PR', 'indice_desempenho': 58.31}, {'uf': 'MA', 'indice_desempenho': 59.58}, {'uf': 'TO', 'indice_desempenho': 61.07}]


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ordem = df.groupby("regiao", observed=True)["media_notas"].mean().sort_values(ascending=False).index
sns.barplot(data=df, x="regiao", y="media_notas", hue="rede_ensino", order=ordem, palette=CORES_REDE, errorbar=("ci", 95), ax=axes[0])
axes[0].set_title("Média das notas por região e rede (IC 95%)", loc="left", fontweight="bold"); axes[0].set_xlabel(""); axes[0].set_ylabel("Média das notas"); axes[0].set_ylim(50, 75); axes[0].legend(title="Rede", frameon=False)
sns.boxplot(data=df, x="regiao", y="media_notas", order=ORDEM_REGIAO, palette=CORES_REGIAO, ax=axes[1])
axes[1].set_title("Distribuição das notas por região", loc="left", fontweight="bold"); axes[1].set_xlabel(""); axes[1].set_ylabel("")
fig.tight_layout()
salvar(fig, "05_regioes_redes.png")
plt.show()

figura salva em imagens/05_regioes_redes.png


### 9.3 Barras por disciplina — comparação educacional

In [20]:
disc = df.groupby(["disciplina", "rede_ensino"], observed=True)[["media_notas", "taxa_reprovacao"]].mean().reset_index()
ordem_disc = df.groupby("disciplina")["media_notas"].mean().sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=disc, x="disciplina", y="media_notas", hue="rede_ensino", order=ordem_disc, palette=CORES_REDE, ax=axes[0])
axes[0].set_title("Média das notas por disciplina e rede", loc="left", fontweight="bold"); axes[0].set_ylim(50, 75); axes[0].set_xlabel(""); axes[0].legend(title="Rede", frameon=False)
for cont in axes[0].containers: axes[0].bar_label(cont, fmt="%.1f", fontsize=8, padding=2)
sns.barplot(data=disc, x="disciplina", y="taxa_reprovacao", hue="rede_ensino", order=ordem_disc, palette=CORES_REDE, ax=axes[1])
axes[1].set_title("Taxa de reprovação por disciplina e rede", loc="left", fontweight="bold"); axes[1].set_xlabel(""); axes[1].set_ylabel("Taxa de reprovação (%)"); axes[1].legend(title="Rede", frameon=False)
for cont in axes[1].containers: axes[1].bar_label(cont, fmt="%.1f", fontsize=8, padding=2)
fig.tight_layout()
salvar(fig, "06_barras_disciplina.png")
plt.show()
print(df.groupby("disciplina")[["media_notas", "taxa_aprovacao", "taxa_reprovacao"]].mean().sort_values("media_notas").round(2))

figura salva em imagens/06_barras_disciplina.png
            media_notas  taxa_aprovacao  taxa_reprovacao
disciplina                                              
Português         63.13           75.50            17.10
História          64.49           78.66            18.54
Ciências          65.38           77.73            18.65
Geografia         65.75           77.08            18.25
Matemática        66.04           77.98            19.20


### 9.4 Heatmap semestral — evolução temporal por região

In [21]:
tabela = df.pivot_table(index="regiao", columns="periodo", values="media_notas", aggfunc="mean", observed=True)
tabela = tabela[sorted(tabela.columns, key=lambda p: (int(p[:4]), int(p[-1])))]
fig, ax = plt.subplots(figsize=(15, 4.5))
sns.heatmap(tabela, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=0.4, linecolor="white", annot_kws={"size": 8}, cbar_kws={"shrink": 0.8, "label": "Média das notas"}, ax=ax)
ax.set_title("Heatmap semestral — média das notas por região (2015–2024)", loc="left", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Período (ano/semestre)"); ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
salvar(fig, "07_heatmap_semestral.png")
plt.show()

figura salva em imagens/07_heatmap_semestral.png


### 9.5 Dispersão renda × notas — correlação

In [22]:
from utils.dados import correlacao_com_significancia, interpretar_correlacao

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (xcol, titulo) in zip(axes, [("renda_media_familiar", "Renda média familiar (R$)"), ("acesso_internet", "Acesso à internet (%)")]):
    sns.scatterplot(data=df, x=xcol, y="media_notas", hue="rede_ensino", palette=CORES_REDE, alpha=0.6, ax=ax)
    sns.regplot(data=df, x=xcol, y="media_notas", scatter=False, color="#333", line_kws={"linestyle": "--"}, ax=ax)
    est = correlacao_com_significancia(df, xcol, "media_notas")
    ax.set_title(f"{titulo} × média das notas\nr = {est['r']:+.3f} ({interpretar_correlacao(est['r'])}), p ≈ {est['p_aprox']:.3f}", loc="left", fontweight="bold")
    ax.set_xlabel(titulo); ax.set_ylabel("Média das notas"); ax.legend(title="Rede", frameon=False)
fig.tight_layout()
salvar(fig, "08_dispersao_renda_internet.png")
plt.show()

figura salva em imagens/08_dispersao_renda_internet.png


In [23]:
matriz = df[numericas].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(matriz, mask=np.triu(np.ones_like(matriz, dtype=bool)), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Matriz de correlação de Pearson", loc="left", fontsize=14, fontweight="bold", pad=12)
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
fig.tight_layout()
salvar(fig, "09_matriz_correlacao.png")
plt.show()
matriz.round(3)

figura salva em imagens/09_matriz_correlacao.png


,media_notas,taxa_aprovacao,taxa_reprovacao,acesso_internet,renda_media_familiar,indice_desempenho
media_notas,1.000,0.007,0.037,0.006,0.007,0.063
taxa_aprovacao,0.007,1.000,0.097,0.033,0.063,0.013
taxa_reprovacao,0.037,0.097,1.000,0.003,-0.041,0.029
acesso_internet,0.006,0.033,0.003,1.000,0.031,0.006
renda_media_familiar,0.007,0.063,-0.041,0.031,1.000,-0.015
indice_desempenho,0.063,0.013,0.029,0.006,-0.015,1.000


In [24]:
por_renda = df.groupby(["faixa_renda", "rede_ensino"], observed=True)["media_notas"].mean().unstack()
por_net = df.groupby(["faixa_internet", "rede_ensino"], observed=True)["media_notas"].mean().unstack()
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
por_renda.plot.bar(ax=axes[0], color=[CORES_REDE[c] for c in por_renda.columns]); axes[0].set_title("Média das notas por quartil de renda", loc="left", fontweight="bold"); axes[0].set_xlabel(""); axes[0].set_ylim(55, 72); axes[0].tick_params(axis="x", rotation=0)
por_net.plot.bar(ax=axes[1], color=[CORES_REDE[c] for c in por_net.columns]); axes[1].set_title("Média das notas por faixa de acesso à internet", loc="left", fontweight="bold"); axes[1].set_xlabel(""); axes[1].set_ylim(55, 72); axes[1].tick_params(axis="x", rotation=0)
for ax in axes:
    ax.legend(title="Rede", frameon=False)
    for cont in ax.containers: ax.bar_label(cont, fmt="%.1f", fontsize=8, padding=2)
fig.tight_layout()
salvar(fig, "10_faixas_renda_internet.png")
plt.show()

figura salva em imagens/10_faixas_renda_internet.png


### 9.6 Tabela dinâmica — exploração detalhada

In [25]:
pivo = pd.pivot_table(df, index=["regiao", "rede_ensino"], columns="disciplina", values="media_notas", aggfunc="mean", observed=True, margins=True, margins_name="Total").round(1)
pivo.style.background_gradient(cmap="YlGnBu", axis=None).format("{:.1f}")

## 10. Interpretação dos resultados

A seguir, cada pergunta orientadora é respondida com base nos resultados acima. Os números são calculados na célula seguinte para garantir coerência com a base.

In [26]:
rank_uf = df.groupby("uf")["indice_desempenho"].mean().sort_values(ascending=False)
rank_reg = df.groupby("regiao", observed=True)["media_notas"].mean().sort_values(ascending=False)
redes = df.groupby("rede_ensino")[["media_notas", "taxa_aprovacao", "indice_desempenho"]].mean()
disc_media = df.groupby("disciplina")["media_notas"].mean().sort_values()
anual = df.groupby("ano")["media_notas"].mean()
coef_ano = np.polyfit(anual.index, anual.values, 1)[0]
r_renda = correlacao_com_significancia(df, "renda_media_familiar", "media_notas")
r_net = correlacao_com_significancia(df, "acesso_internet", "media_notas")
r_apr = correlacao_com_significancia(df, "taxa_aprovacao", "media_notas")
q_renda = df.groupby("faixa_renda", observed=True)["media_notas"].mean()
cv_regiao = (df.groupby("regiao", observed=True)["media_notas"].std() / df.groupby("regiao", observed=True)["media_notas"].mean() * 100).round(1)

print("1) Melhores estados (índice):", rank_uf.head(3).round(2).to_dict(), "| piores:", rank_uf.tail(3).round(2).to_dict())
print(f"2) Privada - Pública: notas {redes.loc['Privada','media_notas'] - redes.loc['Pública','media_notas']:+.2f} | aprovação {redes.loc['Privada','taxa_aprovacao'] - redes.loc['Pública','taxa_aprovacao']:+.2f} p.p.")
print(f"3) Tendência anual das notas: {coef_ano:+.3f} ponto/ano | {anual.index[0]}: {anual.iloc[0]:.2f} -> {anual.index[-1]}: {anual.iloc[-1]:.2f}")
print(f"4) Renda x notas: r = {r_renda['r']:+.3f} (p ≈ {r_renda['p_aprox']:.3f}); Q4 - Q1 = {q_renda.iloc[-1] - q_renda.iloc[0]:+.2f} pontos")
print(f"5) Internet x notas: r = {r_net['r']:+.3f} (p ≈ {r_net['p_aprox']:.3f})")
print(f"6) Disciplinas (menor -> maior nota): {disc_media.round(2).to_dict()}")
print(f"7) Regiões (notas): {rank_reg.round(2).to_dict()} | amplitude {rank_reg.iloc[0] - rank_reg.iloc[-1]:.2f} | CV por região (%): {cv_regiao.to_dict()}")
print(f"Extra) Aprovação x notas: r = {r_apr['r']:+.3f}")

1) Melhores estados (índice): {'RO': 70.49, 'PA': 69.59, 'BA': 68.71} | piores: {'TO': 61.07, 'MA': 59.58, 'PR': 58.31}
2) Privada - Pública: notas -0.15 | aprovação +0.57 p.p.
3) Tendência anual das notas: +0.177 ponto/ano | 2015: 65.48 -> 2024: 66.01
4) Renda x notas: r = +0.007 (p ≈ 0.850); Q4 - Q1 = +0.52 pontos
5) Internet x notas: r = +0.006 (p ≈ 0.881)
6) Disciplinas (menor -> maior nota): {'Português': 63.13, 'História': 64.49, 'Ciências': 65.38, 'Geografia': 65.75, 'Matemática': 66.04}
7) Regiões (notas): {'Centro-Oeste': 68.85, 'Sul': 66.02, 'Nordeste': 64.98, 'Sudeste': 63.69, 'Norte': 63.17} | amplitude 5.68 | CV por região (%): {'Norte': 25.0, 'Nordeste': 29.1, 'Centro-Oeste': 25.8, 'Sudeste': 26.9, 'Sul': 26.2}
Extra) Aprovação x notas: r = +0.007


**1. Quais estados apresentam melhor desempenho?** O ranking pelo índice médio de desempenho tem amplitude de poucos pontos entre o primeiro e o último colocado. Os três melhores e os três piores estados aparecem na saída acima e no gráfico 9.2. Não há agrupamento regional claro no topo ou na base do ranking — estados de regiões diferentes se alternam — o que já antecipa a resposta da pergunta 7.

**2. Existem diferenças entre escolas públicas e privadas?** A diferença de nota média entre as redes é de apenas 0,15 ponto (rede pública ligeiramente à frente) e a diferença de aprovação é de 0,6 ponto percentual. Nos boxplots, as distribuições se sobrepõem quase integralmente. Nesta base, portanto, a rede de ensino **não** é um fator discriminante — diferentemente do que se observa em avaliações reais (SAEB/ENEM), nas quais a rede privada supera a pública com folga.

**3. Houve melhoria ao longo do tempo?** A tendência linear da média anual é praticamente nula (poucos centésimos de ponto por ano) e a série semestral oscila em torno de 65 pontos, alternando anos melhores (2023, 2019, 2017) e piores (2018, 2016, 2022). O período da pandemia (2020–2021) não apresenta queda destacada. Conclusão: **estabilidade**, sem melhoria consistente.

**4. Existe relação entre renda e desempenho?** A correlação de Pearson entre renda média familiar e média das notas é praticamente nula e não significativa (p > 0,05). A diferença de notas entre o quartil mais rico e o mais pobre é desprezível. A simulação não reproduz a relação positiva esperada na literatura.

**5. Como o acesso à internet impacta o aprendizado?** Também aqui a correlação é praticamente nula. As faixas de acesso digital apresentam médias de notas muito próximas, nas duas redes.

**6. Quais disciplinas apresentam menor rendimento?** **Português** é a disciplina mais crítica (menor nota média), seguida de História; Matemática tem a maior média. O resultado inverte o padrão real (em que Matemática costuma ser a disciplina crítica), mais um indício do caráter aleatório da base.

**7. Existem desigualdades regionais relevantes?** As médias regionais diferem em cerca de 5,7 pontos entre a melhor região (Centro-Oeste, ≈ 68,9) e a pior (Norte, ≈ 63,2), com Sul, Nordeste e Sudeste no meio. É a maior diferença entre grupos encontrada na base, mas ainda menor do que a dispersão **dentro** de cada região (desvio-padrão em torno de 17 pontos; coeficiente de variação entre 25 % e 29 %). Ou seja: a heterogeneidade interna supera a diferença entre regiões. O padrão também difere do real, em que Sul/Sudeste lideram e Norte/Nordeste ficam atrás.

**Observação sobre a qualidade dos dados.** Duas inconsistências foram documentadas na seção 5: aprovação + reprovação > 100 % em cerca de 40 % dos registros e um `nivel_desempenho` independente do `indice_desempenho`. Além disso, a correlação entre taxa de aprovação e média das notas é praticamente zero — em dados reais ela seria fortemente positiva. Essas evidências convergem: trata-se de uma base **simulada com variáveis geradas de forma independente**. O valor do projeto está no método (pipeline reprodutível, KPIs, visualizações, dashboard), e não em conclusões sobre a educação brasileira.

## 11. Conclusão

- **Pipeline reprodutível:** leitura → tratamento (tipos, textos, duplicatas, imputação, limites, flags de consistência) → engenharia de atributos (período, faixas de renda e internet, nível recalculado, score composto) → persistência em SQLite com modelo relacional → KPIs e visualizações. As mesmas funções (`utils/dados.py`) alimentam o dashboard Streamlit, garantindo coerência entre notebook e aplicação.
- **KPIs nacionais:** média das notas ≈ 65, taxa de aprovação ≈ 77 %, índice de desempenho ≈ 65; o estado líder, a rede líder e a disciplina crítica estão na tabela da seção 8.
- **Principais achados:** estabilidade temporal (+0,18 ponto/ano), diferença mínima entre redes (0,15 ponto, com a rede pública ligeiramente à frente), correlações nulas de renda e internet com notas, Português como disciplina mais crítica, Centro-Oeste à frente e Norte atrás (5,7 pontos) e desigualdade intrarregional maior do que a inter-regional.
- **Limitações:** base simulada com inconsistências internas documentadas; os achados demonstram o método analítico e não devem ser lidos como diagnóstico da educação brasileira.
- **Próximos passos:** aplicar o mesmo pipeline a dados reais (SAEB/IDEB/Censo Escolar via APIs do INEP/IBGE), incluir variáveis de infraestrutura escolar e formação docente, e evoluir para modelos preditivos de risco de baixo desempenho.

O dashboard interativo está em `app.py` (+ `pages/`), publicado no Streamlit Community Cloud, e a página de apresentação do projeto em `index.html` (GitHub Pages).